# 02 - PDF NER (Task 2: clean PDFs, Task 3: scanned PDFs)

Pipeline: PDF -> PyMuPDF -> text -> GLiNER (`gliner_small-v2.5`) -> predictions -> strict / relaxed evaluation.
The gold annotations are those of `data/plain/validation.jsonl`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))  # makes `import ner` work from /notebooks

from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.inference import (
    LABEL_DESCRIPTIONS,
    get_dataset_labels,
    load_gliner_model,
    predict_dataset,
)
from ner.pdf import extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl

VALIDATION_PATH = ROOT / "data" / "plain" / "validation.jsonl"
PDF_CLEAN_DIR = ROOT / "data" / "pdf_clean"
PDF_SCANNED_DIR = ROOT / "data" / "pdf_scanned"
RESULTS_CLEAN = ROOT / "results" / "pdf_clean" / "gliner_small_validation.jsonl"
RESULTS_SCANNED = ROOT / "results" / "pdf_scanned" / "gliner_small_validation.jsonl"
RESULTS_SCANNED_OCR = ROOT / "results" / "pdf_scanned" / "gliner_small_ocr_validation.jsonl"
THRESHOLD = 0.5

In [ ]:
gold = load_jsonl(VALIDATION_PATH)
assert gold, f"{VALIDATION_PATH} is empty: copy the course data/ folder into the project first."
LABELS = get_dataset_labels(gold)
print(f"{len(gold)} gold documents | labels: {LABELS}")

assert any(PDF_CLEAN_DIR.glob("*.pdf")), f"No PDFs in {PDF_CLEAN_DIR}"
assert any(PDF_SCANNED_DIR.glob("*.pdf")), f"No PDFs in {PDF_SCANNED_DIR}"

## PyMuPDF quick start: extracted text vs. gold text

In [ ]:
from ner.pdf import extract_text_from_pdf, resolve_pdf_path

doc_id = gold[0]["id"]
clean_text = extract_text_from_pdf(resolve_pdf_path(PDF_CLEAN_DIR, doc_id))
scanned_text = extract_text_from_pdf(resolve_pdf_path(PDF_SCANNED_DIR, doc_id))

print(f"gold: {len(gold[0]['text'])} chars | clean PDF: {len(clean_text)} chars | scanned PDF: {len(scanned_text)} chars")
print("--- clean ---")
print(clean_text[:400])
print("--- scanned ---")
print(repr(scanned_text[:400]))

In [ ]:
model = load_gliner_model("gliner-community/gliner_small-v2.5")

## Task 2 - Clean PDFs

In [ ]:
clean_docs = extract_pdf_dataset(PDF_CLEAN_DIR, gold)
preds = predict_dataset(model, clean_docs, labels=LABELS, threshold=THRESHOLD)
save_responses_to_jsonl(preds, RESULTS_CLEAN)

strict_clean = evaluate_ner_strict(VALIDATION_PATH, RESULTS_CLEAN)
relaxed_clean = evaluate_ner_relaxed(VALIDATION_PATH, RESULTS_CLEAN)

## Task 3 - Scanned PDFs

Same pipeline on `data/pdf_scanned/`. Scanned pages may have no (or a noisy) text layer, so the number of
empty documents is printed.

In [ ]:
scanned_docs = extract_pdf_dataset(PDF_SCANNED_DIR, gold)
print("empty documents:", sum(1 for d in scanned_docs if not d["text"].strip()), "/", len(scanned_docs))

preds = predict_dataset(model, scanned_docs, labels=LABELS, threshold=THRESHOLD)
save_responses_to_jsonl(preds, RESULTS_SCANNED)

strict_scanned = evaluate_ner_strict(VALIDATION_PATH, RESULTS_SCANNED)
relaxed_scanned = evaluate_ner_relaxed(VALIDATION_PATH, RESULTS_SCANNED)

### Optional - scanned PDFs with OCR

Needs Tesseract installed on the system (see the PyMuPDF OCR documentation). If it is not available the
cell reports the error and the rest of the notebook is unaffected.

In [ ]:
try:
    ocr_docs = extract_pdf_dataset(PDF_SCANNED_DIR, gold, use_ocr=True)
    preds = predict_dataset(model, ocr_docs, labels=LABELS, threshold=THRESHOLD)
    save_responses_to_jsonl(preds, RESULTS_SCANNED_OCR)
    strict_ocr = evaluate_ner_strict(VALIDATION_PATH, RESULTS_SCANNED_OCR)
    relaxed_ocr = evaluate_ner_relaxed(VALIDATION_PATH, RESULTS_SCANNED_OCR)
except Exception as exc:  # Tesseract missing, etc.
    print(f"OCR skipped: {type(exc).__name__}: {exc}")

## Results summary (all experiments found in `results/`)

In [ ]:
from ner.report import build_results_table

table, missing = build_results_table(ROOT)
print(table)
if missing:
    print("\nNot evaluated yet:", *missing, sep="\n  - ")